# Tool-Router Ladder: Stage E3, DPO from the SFT adapter

`src/train_dpo.py`: the SFT LoRA is trained further on (teacher action, student mistake) pairs built
on the Mac by `make pairs` (execution-equivalent calls already removed). It refuses to start unless
the parent adapter's weights are in `configs/sft_registry.yaml` and the pairs were built from that
adapter's completions.

**Run this only after you approve the pairs (E2-4).**

**Settings:** Accelerator **GPU T4 x2**, Internet **on**, the pairs dataset for ONE model attached
(upload `data/pairs/<model>/` as a Kaggle dataset).
**Secrets:** `HF_TOKEN` (write) and `WANDB_API_KEY`, both ticked.
**Run:** Save Version -> **Save & Run All (Commit)**. The reference log-probs and each epoch are
checkpointed; a rerun resumes. **No API quota.**

**2026-10-26, run 2:** the first run collapsed (the model stopped calling tools). This run adds the
chosen-side NLL anchor and balanced decision pairs (`configs/v2/dpo.yaml`), pushes to a fresh repo
`tool-router-<model>-dpo2`, and refuses to push if the first-token call rate collapses.

**Send back:** the smoke cell's last lines and the two `first-token call rate` lines, and the final `[train_dpo] done: ...` line.

In [ ]:
# Secrets: Add-ons -> Secrets, and TICK each one for this notebook.
import os
from kaggle_secrets import UserSecretsClient
_s = UserSecretsClient()
for key in ["HF_TOKEN", "WANDB_API_KEY"]:
    os.environ[key] = _s.get_secret(key)
print("secrets loaded:", ["HF_TOKEN", "WANDB_API_KEY"])

In [ ]:
# Step out of the repo BEFORE deleting it: rmtree on the kernel's own working
# directory leaves cwd pointing at a dead inode, and then every ! command fails
# with "getcwd: cannot access parent directories" -- including this git clone.
%cd /kaggle/working
import shutil
shutil.rmtree("/kaggle/working/the_llm_project", ignore_errors=True)
!git clone -q https://github.com/madhusiddharths/the_llm_project.git /kaggle/working/the_llm_project
%cd /kaggle/working/the_llm_project
!git log -1 --oneline

!pip install -q peft
# Kaggle ships torchao 0.10.0; peft's LoRA dispatcher raises on anything below
# 0.16.0 instead of skipping it. Nothing here uses torchao, and with the package
# gone peft's is_torchao_available() returns False and dispatch falls through.
!pip uninstall -y -q torchao

In [ ]:
# Finds the attached pairs dataset (make_pairs.py output: pairs.jsonl + manifest.json).
import glob, json, os
hits = [h for h in glob.glob("/kaggle/input/**/manifest.json", recursive=True)
        if json.load(open(h)).get("file") == "pairs.jsonl"]
assert hits, "attach the tool-router-pairs dataset (Add Input): no pairs manifest under /kaggle/input"
PAIRS = os.path.dirname(hits[0])
m = json.load(open(hits[0]))
print("PAIRS =", PAIRS); print("adapter:", m["adapter"], "| pairs:", m["n_pairs"], m["pairs_per_bucket"])
HF_USER = "madhusiddharths1"
MODEL = m["adapter"].split("tool-router-")[1].removesuffix("-sft")   # qwen05b / qwen15b / qwen3b
SHARD = "--shard" if MODEL == "qwen3b" else ""                       # 3B was trained sharded
print("MODEL =", MODEL, SHARD)

### 1. Smoke: one step on the longest pair (memory check; no Hub push)

In [ ]:
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python src/train_dpo.py --config configs/v2/{MODEL}-dpo.yaml --data "$PAIRS" --parent "$HF_USER/tool-router-{MODEL}-sft" --output-dir /kaggle/working/dpo-smoke --smoke {SHARD}

### 2. The run (2 epochs, 8 pairs per step)

In [ ]:
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python src/train_dpo.py --config configs/v2/{MODEL}-dpo.yaml --data "$PAIRS" --parent "$HF_USER/tool-router-{MODEL}-sft" --output-dir /kaggle/working/{MODEL}-dpo2 --hub-repo "$HF_USER/tool-router-{MODEL}-dpo2" --wandb {SHARD}